# PaceAlgo ML — Notebook 04: Triple Barrier Labeling

**Zweck:** Pro Bar generieren wir einen ML-Label (+1/0/-1) basierend auf der Triple-Barrier-Methode (Marcos López de Prado):
- **+1 (WIN):** TP-Barrier wird zuerst getroffen
- **-1 (LOSS):** SL-Barrier wird zuerst getroffen
- **0 (NEUTRAL):** Time-Barrier zuerst getroffen (keine klare Bewegung)

**Barriere-Definition (long-Perspektive):**
- Entry: close[t]
- SL: entry - 1.0 × ATR(t)
- TP: entry + tp_R × 1.0 × ATR(t)
- Time: t + 24 bars (= 2h auf 5M, 6h auf 15M)

**Wir testen drei R-Werte: 1.5, 2.0, 2.5** — pro Asset-Cluster wählen wir später den datengetriebenen Sweet Spot.

**Output:** `/content/processed/labels_<symbol>_<tf>_R<X>.parquet` — pro Bar ein Label.

**Laufzeit:** ~5-10 Min für alle 14 (symbol, tf) × 3 R-Werte = 42 Label-Sets.

## 1. Environment Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_RAW = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_RAW, DATA_PROCESSED, ARTIFACTS_REPORTS
    ARTIFACTS = ARTIFACTS_REPORTS.parent

ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Raw OHLCV (read):   {DATA_RAW}')
print(f'Labels out (write): {DATA_PROCESSED}')

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from core.config import (
    CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES,
    TRIPLE_BARRIER_R_MULTIPLES,
    TRIPLE_BARRIER_SL_ATR_MULT,
    TRIPLE_BARRIER_TIME_LIMIT_BARS,
)
from core.labeling import (
    compute_triple_barrier_labels,
    label_distribution,
    expected_value_per_label_set,
)

ALL_SYMBOLS = CRYPTO_SYMBOLS + FX_SYMBOLS + METAL_SYMBOLS
R_VALUES = TRIPLE_BARRIER_R_MULTIPLES  # [1.5, 2.0, 2.5]
SL_ATR_MULT = TRIPLE_BARRIER_SL_ATR_MULT  # 1.0
TIME_BARS = TRIPLE_BARRIER_TIME_LIMIT_BARS  # 24

print(f'Will compute labels for {len(ALL_SYMBOLS)} symbols × {len(PRIMARY_TIMEFRAMES)} TFs × {len(R_VALUES)} R-values')
print(f'= {len(ALL_SYMBOLS) * len(PRIMARY_TIMEFRAMES) * len(R_VALUES)} label sets')
print(f'R values: {R_VALUES}, SL ATR mult: {SL_ATR_MULT}, time limit: {TIME_BARS} bars')

## 2. Load Asset Clusters (from NB 03)

In [ ]:
clusters_path = ARTIFACTS / 'asset_clusters.parquet'
if clusters_path.exists():
    clusters = pd.read_parquet(clusters_path)
    print(f'Loaded {len(clusters)} cluster assignments')
    display(clusters[['symbol','tf','cluster','cluster_label']])
else:
    print('WARNING: asset_clusters.parquet not found. Run notebook 03 first.')
    clusters = pd.DataFrame()

## 3. Compute Triple Barrier Labels

In [ ]:
def load_ohlcv(symbol: str, tf: str):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    if not p.exists():
        return None
    return pd.read_parquet(p)

summary_rows = []
n_combos = len(ALL_SYMBOLS) * len(PRIMARY_TIMEFRAMES) * len(R_VALUES)
with tqdm(total=n_combos, desc='Label sets') as pbar:
    for symbol in ALL_SYMBOLS:
        for tf in PRIMARY_TIMEFRAMES:
            ohlcv = load_ohlcv(symbol, tf)
            if ohlcv is None or ohlcv.empty:
                print(f'  SKIP {symbol} {tf}: no OHLCV')
                pbar.update(len(R_VALUES))
                continue

            for R in R_VALUES:
                out_path = DATA_PROCESSED / f'labels_{symbol}_{tf}_R{R}.parquet'
                if out_path.exists():
                    print(f'  skip {symbol} {tf} R={R} (cached)')
                    pbar.update(1)
                    continue

                labels = compute_triple_barrier_labels(
                    ohlcv, tp_R=R, sl_atr_mult=SL_ATR_MULT,
                    time_barrier_bars=TIME_BARS,
                )
                if labels.empty:
                    print(f'  EMPTY {symbol} {tf} R={R}')
                    pbar.update(1)
                    continue

                labels.to_parquet(out_path, compression='zstd')

                stats = expected_value_per_label_set(labels, tp_R=R, sl_atr_mult=SL_ATR_MULT)
                summary_rows.append({
                    'symbol': symbol, 'tf': tf, 'R': R,
                    'n': stats['n'],
                    'wins': stats['wins'],
                    'losses': stats['losses'],
                    'neutrals': stats['neutrals'],
                    'win_rate': stats['win_rate'],
                    'expected_R': stats['expected_R'],
                    'profit_factor': stats['profit_factor'],
                })
                pbar.update(1)

print('\nDone.')

## 4. Random-Entry Baseline Stats

Diese Stats zeigen wie ein **random Long-Entry** bei jeder Bar performt — ohne ML-Filter. Pro Asset/TF/R sehen wir Win-Rate, Expected R-Multiple und Profit Factor.

**Theoretische Baseline ohne Edge:** WR ≈ 1/(1+R) → R=1.5 → ~40%, R=2.0 → ~33%, R=2.5 → ~29%.
Profit Factor sollte ohne Edge bei ~1.0 liegen.

**Das Modell muss Edge erzeugen** durch selektive Entry-Wahl → wir wollen PF > 1.5 nach Filterung.

In [ ]:
# Rebuild summary from disk so it works on resume
label_files = sorted(DATA_PROCESSED.glob('labels_*.parquet'))
rebuilt = []
for p in label_files:
    # Parse filename: labels_BTCUSDT_5m_R1.5.parquet
    parts = p.stem.split('_')
    sym = parts[1]
    tf = parts[2]
    R = float(parts[3][1:])
    labels = pd.read_parquet(p)
    stats = expected_value_per_label_set(labels, tp_R=R, sl_atr_mult=SL_ATR_MULT)
    rebuilt.append({
        'symbol': sym, 'tf': tf, 'R': R,
        'n': stats['n'], 'wins': stats['wins'], 'losses': stats['losses'],
        'neutrals': stats['neutrals'], 'win_rate': stats['win_rate'],
        'expected_R': stats['expected_R'], 'profit_factor': stats['profit_factor'],
    })
sdf = pd.DataFrame(rebuilt)

# Attach cluster labels
if not clusters.empty:
    sdf = sdf.merge(clusters[['symbol','tf','cluster_label']], on=['symbol','tf'], how='left')

print(f'Total label sets on disk: {len(sdf)}')
print('\nDescriptive stats per (symbol, tf, R):')
display(sdf.round(4))

## 5. Aggregate per Cluster — Which R is Best per Asset Class?

In [ ]:
if 'cluster_label' in sdf.columns:
    agg = sdf.groupby(['cluster_label', 'R']).agg(
        avg_win_rate=('win_rate', 'mean'),
        avg_exp_R=('expected_R', 'mean'),
        avg_PF=('profit_factor', 'mean'),
        total_n=('n', 'sum'),
    ).round(4).reset_index()
    print('Per-cluster random-entry baseline (NO ML filter applied):')
    display(agg)
    print('\nNote: PF close to 1.0 = no edge from random entries — expected.')
    print('Our ML job: filter entries to push PF > 1.5 with appropriate R per cluster.')
else:
    print('No cluster labels available (NB 03 not run?)')

## 6. Sanity Check — BTC 5M R=1.5 Sample

In [ ]:
sample_path = DATA_PROCESSED / 'labels_BTCUSDT_5m_R1.5.parquet'
if sample_path.exists():
    sample = pd.read_parquet(sample_path)
    print(f'Sample shape: {sample.shape}')
    print(f'Label distribution:')
    print(sample['label'].value_counts(dropna=True).sort_index())
    print(f'\nHit-type distribution:')
    print(sample['hit_type'].value_counts())
    print(f'\nHit offset stats (bars until label assigned):')
    print(sample['hit_bar_offset'].describe())
    print(f'\nSample tail:')
    display(sample.tail())

## 7. Sync to Drive Backup

In [ ]:
if IS_COLAB:
    drive_target = DRIVE_BACKUP
    drive_target.mkdir(parents=True, exist_ok=True)
    print(f'Syncing labels to {drive_target}')
    !rsync -ah {DATA_PROCESSED}/labels_*.parquet {drive_target}/
    print('Sync done.')

---

## Nächster Schritt

→ `05_train_lgbm.ipynb`

Erstes ML-Training: LightGBM auf den Features + Labels. Walk-Forward Validation, SHAP-Analyse, Confusion Matrix. Wir werden sehen ob das Modell echte Edge generieren kann (PF > 1.5 OOS) oder ob wir die Architektur überarbeiten müssen.